# Running Models — Audio · Part 8 — Companion Notebook

This goes with **"Score Your Own Scene: A Prompt-to-Soundtrack App"** — the series finale. We generate a music bed with MusicGen and a narration line with SpeechT5, resample them to a common rate, mix and normalize into one clip, and wrap it in a Gradio app.

Run cells top to bottom (Shift+Enter). **MusicGen wants a GPU:** Runtime → Change runtime type → GPU.

## Install

The union of everything in the series — both models, the speaker dataset, the audio + app libs.

In [ ]:
!pip install -q transformers datasets soundfile gradio sentencepiece

## Load both halves of the series

Part 6's MusicGen and Part 2's SpeechT5, side by side. Note we read the music rate from the config rather than hard-coding it.

In [ ]:
import numpy as np, torch
import torchaudio.functional as AF
from transformers import (
    AutoProcessor, MusicgenForConditionalGeneration,
    SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan,
)
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"

mg_proc = AutoProcessor.from_pretrained("facebook/musicgen-small")
mg = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small").to(device)
MUSIC_SR = mg.config.audio_encoder.sampling_rate     # 32000

tts_proc = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
speaker = torch.tensor(
    load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")[7306]["xvector"]
).unsqueeze(0)

## The whole series, in one function

Generate, **resample** (the crux — you can only add waveforms that share a sample rate), mix with the music ducked under the voice, then normalize so the sum doesn't clip.

In [ ]:
def soundtrack(mood: str, line: str):
    # 1. music bed (MusicGen, 32 kHz, mono)
    mi = mg_proc(text=[mood], padding=True, return_tensors="pt").to(device)
    music = mg.generate(**mi, max_new_tokens=512)[0, 0].cpu()          # ~10 s

    # 2. narration (SpeechT5, 16 kHz) -> resample up to the music rate
    ni = tts_proc(text=line, return_tensors="pt")
    voice = tts.generate_speech(ni["input_ids"], speaker, vocoder=vocoder)
    voice = AF.resample(voice, 16000, MUSIC_SR)                        # <-- the key step

    # 3. mix: music ducked under the voice, then normalize
    out = music * 0.5
    n = min(len(out), len(voice))
    out[:n] = out[:n] + voice[:n] * 0.9
    out = out / out.abs().max()                                       # avoid clipping
    return MUSIC_SR, out.numpy()

## Try it

A mood and a line in, a scored scene out: strings swelling, a voice landing the moment, the music carrying on after.

In [ ]:
sr, audio = soundtrack("warm cinematic strings, hopeful", "And so, the journey finally began.")
from IPython.display import Audio
Audio(audio, rate=sr)

## Wrap it in Gradio

Two text boxes in (a list, passed to `mood` then `line` in order), an audio player out. `.launch()` prints a public link.

In [ ]:
import gradio as gr

gr.Interface(
    fn=soundtrack,
    inputs=[gr.Textbox(label="Mood / style"), gr.Textbox(label="Narration line")],
    outputs=gr.Audio(label="Your scene"),
    title="Score Your Own Scene",
).launch()

---

That's the series. From "audio is a tensor" to scoring a scene from two text boxes. The next step — making a model better at *your* sound — is fine-tuning, which is the **Training Models** (phase 2) series.